# 📊 Base Laya-Multilingual Benchmark on 3,000 Fatwa Decisions
### Measuring Zero-Shot Baseline Performance to Quantify Fine-Tuning Improvements

This notebook runs the untuned, off-the-shelf **`convaiinnovations/laya-multilingual`** (322M mmBERT System 1 engine) across all **3,000 official Islamic Fatwa relevance examples** labeled by GPT-4o Mini.

---

### Objectives
1. **Establish the Zero-Shot Baseline**: See how well the raw pre-trained multilingual model performs on Arabic jurisprudential relevance without any task-specific fine-tuning.
2. **Quantify the Lift**: Generate exact baseline metrics (Accuracy, Precision, Recall, F1, Per-Scholar Accuracy, and Latency) to directly compare against your fine-tuned model from Notebook 07.
3. **Execution Time**: Takes **< 1 minute** on a Kaggle GPU (T4 or P100)!


In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Target Device: [{device.upper()}]")


In [ ]:
# 2. Install Dependencies
!pip install -q "laya>=0.1.6"

import laya
import transformers
print(f"✓ Laya: {laya.__version__} | Transformers: {transformers.__version__}")


In [ ]:
# 3. Locate & Load the 3,000 Official Labeled Fatwa Candidates
import os
import json
import zipfile
from pathlib import Path
import pandas as pd

LABELED_DATA = None
search_paths = [
    Path("/kaggle/working/candidates_6k_bundle.zip"),
    Path("/kaggle/working/official_labeled_3k.json"),
    Path("/kaggle/working/official_labeled_3k.csv"),
    Path("outputs/candidates_6k_bundle.zip"),
    Path("outputs/official_labeled_3k.json"),
    Path("data/candidates_6k_bundle.zip"),
    Path("data/official_labeled_3k.json"),
]

# Check Kaggle input folders as well
for p in Path("/kaggle/input").rglob("*"):
    if p.name in ("candidates_6k_bundle.zip", "official_labeled_3k.json", "official_labeled_3k.csv"):
        search_paths.insert(0, p)

for sp in search_paths:
    if sp.exists():
        LABELED_DATA = sp
        break

assert LABELED_DATA is not None, "Could not find official_labeled_3k.json or candidates_6k_bundle.zip! Please add candidates_6k_bundle.zip to Input."
print(f"✓ Found dataset source at: {LABELED_DATA}")

raw_items = []
if LABELED_DATA.name.endswith(".zip"):
    with zipfile.ZipFile(LABELED_DATA, "r") as zf:
        if "official_labeled_3k.json" in zf.namelist():
            with zf.open("official_labeled_3k.json") as f:
                raw_items = json.load(f)
        elif "official_labeled_3k.csv" in zf.namelist():
            with zf.open("official_labeled_3k.csv") as f:
                raw_items = pd.read_csv(f).to_dict(orient="records")
elif LABELED_DATA.name.endswith(".json"):
    with open(LABELED_DATA, "r", encoding="utf-8") as f:
        raw_items = json.load(f)
        if isinstance(raw_items, dict):
            raw_items = list(raw_items.values())
elif LABELED_DATA.name.endswith(".csv"):
    raw_items = pd.read_csv(LABELED_DATA).to_dict(orient="records")

print(f"✓ Successfully loaded {len(raw_items):,} labeled candidates.")

# Filter valid items
valid_items = []
for it in raw_items:
    v = str(it.get("gpt_verdict", it.get("verdict", ""))).strip().lower().replace("_", "-")
    if v in ("answers-it", "does-not-answer"):
        it["gold_label"] = v
        valid_items.append(it)

print(f"✓ Valid candidates ready for evaluation: {len(valid_items):,}")
pos_count = sum(1 for it in valid_items if it["gold_label"] == "answers-it")
print(f"  • Positives ('answers-it'):     {pos_count:,} ({100*pos_count/len(valid_items):.1f}%)")
print(f"  • Negatives ('does-not-answer'): {len(valid_items)-pos_count:,} ({100*(len(valid_items)-pos_count)/len(valid_items):.1f}%)")


In [ ]:
# 4. Load Base (Untuned) Laya-Multilingual on GPU
MODEL_ID = "convaiinnovations/laya-multilingual"
print(f"Loading base zero-shot model [{MODEL_ID}] on [{device.upper()}]...")

base_agent = laya.load(MODEL_ID, device=device)
print("✓ Base Laya Multilingual loaded and ready for zero-shot inference!")


In [ ]:
# 5. Evaluate All 3,000 Candidates with Base Laya
import time
import numpy as np
from tqdm.auto import tqdm

QUESTION_SCHEMA = {
    "relevance": {
        "type": "choice",
        "instructions": (
            "Determine whether the Islamic fatwa directly and specifically answers the user's question.\n"
            "- 'answers-it': The fatwa directly and specifically answers the question asked.\n"
            "- 'does-not-answer': The fatwa does NOT directly answer the question (touches on a related topic or is unrelated)."
        ),
        "criteria": {
            "answers-it": "The fatwa directly answers the specific question asked.",
            "does-not-answer": "The fatwa does not directly answer the question.",
        }
    }
}

results = []
latencies_ms = []

print(f"Running zero-shot inference across {len(valid_items):,} examples...")
t0_all = time.time()

for item in tqdm(valid_items, desc="Zero-Shot Base Laya Inference"):
    state = {
        "user_query": str(item.get("question", "")).strip(),
        "scholar": str(item.get("scholar", "")).strip(),
        "fatwa_title": str(item.get("title", "")).strip(),
        "fatwa_answer": str(item.get("answer_text", ""))[:1400].strip(),
    }
    
    t0 = time.perf_counter()
    res = base_agent.predict(state, QUESTION_SCHEMA)
    dt_ms = (time.perf_counter() - t0) * 1000
    latencies_ms.append(dt_ms)
    
    ans = res["answers"].get("relevance", {})
    pred_choice = ans.get("choice", "unknown")
    conf = ans.get("confidence", 0.0)
    probs = ans.get("probabilities", {})
    
    gold = item["gold_label"]
    results.append({
        "id": item.get("id"),
        "scholar": item.get("scholar"),
        "question": item.get("question"),
        "title": item.get("title"),
        "gold_label": gold,
        "base_pred": pred_choice,
        "base_confidence": conf,
        "prob_answers_it": probs.get("answers-it", 0.0),
        "prob_does_not_answer": probs.get("does-not-answer", 0.0),
        "latency_ms": dt_ms,
        "correct": int(pred_choice == gold),
    })

total_eval_time = time.time() - t0_all
print(f"\n✓ Completed all {len(results):,} inferences in {total_eval_time:.2f} seconds!")
print(f"  • Throughput: {len(results)/total_eval_time:.1f} inferences/second")


In [ ]:
# 6. Baseline Performance Report & Scholar Breakdown
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

df_base = pd.DataFrame(results)

overall_acc = df_base["correct"].mean()
p50_lat = np.percentile(latencies_ms, 50)
p95_lat = np.percentile(latencies_ms, 95)
avg_lat = np.mean(latencies_ms)

print("=" * 80)
print("📊 BASE LAYA-MULTILINGUAL (ZERO-SHOT) BENCHMARK REPORT")
print("=" * 80)
print(f"Total Evaluated Items:           {len(df_base):,}")
print(f"Overall Accuracy vs GPT-4o Mini: {overall_acc * 100:.2f}%")
print(f"Median Latency (p50):            {p50_lat:.2f} ms")
print(f"95th Percentile Latency (p95):   {p95_lat:.2f} ms")
print(f"Average Latency:                 {avg_lat:.2f} ms")
print(f"Total Evaluation Time:           {total_eval_time:.2f}s")
print("=" * 80)

print("\nDETAILED CLASSIFICATION REPORT (ZERO-SHOT BASELINE):")
print(classification_report(df_base["gold_label"], df_base["base_pred"], digits=3))

print("\nCONFUSION MATRIX:")
labels = ["answers-it", "does-not-answer"]
cm = confusion_matrix(df_base["gold_label"], df_base["base_pred"], labels=labels)
cm_df = pd.DataFrame(cm, index=[f"Actual {l}" for l in labels], columns=[f"Pred {l}" for l in labels])
display(cm_df)

print("\nPERFORMANCE BREAKDOWN BY SCHOLAR:")
scholar_perf = df_base.groupby("scholar").agg(
    total=("correct", "count"),
    correct=("correct", "sum"),
    accuracy=("correct", "mean"),
    avg_conf=("base_confidence", "mean"),
    avg_latency_ms=("latency_ms", "mean")
).reset_index()

scholar_perf["accuracy"] = scholar_perf["accuracy"].map(lambda x: f"{x*100:.2f}%")
scholar_perf["avg_conf"] = scholar_perf["avg_conf"].map(lambda x: f"{x:.3f}")
scholar_perf["avg_latency_ms"] = scholar_perf["avg_latency_ms"].map(lambda x: f"{x:.2f} ms")
display(scholar_perf)


In [ ]:
# 7. Save Baseline Predictions & Metrics Summary
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save detailed item-by-item predictions
csv_path = OUTPUT_DIR / "base_laya_predictions_3k.csv"
json_path = OUTPUT_DIR / "base_laya_predictions_3k.json"

df_base.to_csv(csv_path, index=False, encoding="utf-8")
df_base.to_json(json_path, orient="records", force_ascii=False, indent=2)

# 2. Save structured metrics JSON for direct comparison
metrics_summary = {
    "model": "convaiinnovations/laya-multilingual (base untuned)",
    "total_samples": len(df_base),
    "accuracy": float(overall_acc),
    "median_latency_ms": float(p50_lat),
    "p95_latency_ms": float(p95_lat),
    "total_time_s": float(total_eval_time),
    "positives_actual": int((df_base["gold_label"] == "answers-it").sum()),
    "negatives_actual": int((df_base["gold_label"] == "does-not-answer").sum()),
    "positives_predicted": int((df_base["base_pred"] == "answers-it").sum()),
    "negatives_predicted": int((df_base["base_pred"] == "does-not-answer").sum()),
    "confusion_matrix": {
        "true_positives": int(cm[0][0]),
        "false_negatives": int(cm[0][1]),
        "false_positives": int(cm[1][0]),
        "true_negatives": int(cm[1][1]),
    }
}

metrics_path = OUTPUT_DIR / "base_laya_benchmark_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)

print("=" * 80)
print("✓ BASELINE ASSETS SAVED TO WORKING DIRECTORY:")
print("=" * 80)
print(f"  • {csv_path.name:32s} ({csv_path.stat().st_size / (1024**2):.2f} MB)")
print(f"  • {json_path.name:32s} ({json_path.stat().st_size / (1024**2):.2f} MB)")
print(f"  • {metrics_path.name:32s} ({metrics_path.stat().st_size / 1024:.2f} KB)")
print("=" * 80)
print("You can download these files or leave them in /kaggle/working to plot the comparison!")
